In [87]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [88]:


GRID = 7

class Binarize(torch.nn.Module):
    def __init__(self, threshold=0.4):
        super().__init__()
        self.threshold = threshold
    def forward(self, x):
        return (x > self.threshold).float()

train_tf = v2.Compose([
    v2.ToImage(),                                # PIL -> tv_tensors.Image (uint8)
    v2.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    v2.Resize((GRID, GRID), antialias=True),
    v2.ToDtype(torch.float32, scale=True),       # -> float in [0, 1]
    Binarize(0.25),
])

test_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize((GRID, GRID), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    Binarize(0.25),
])

train = datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
test  = datasets.MNIST("./data", train=False, download=True, transform=test_tf)

train_dl = DataLoader(train, batch_size=128, shuffle=True,  num_workers=0)
test_dl  = DataLoader(test,  batch_size=256, shuffle=False, num_workers=0)

In [89]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(GRID*GRID, 8),
            nn.ReLU(),
            nn.Linear(8,8),
            nn.ReLU(),
            nn.Linear(8,10),
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.linear_relu_stack(x)

model = NeuralNetwork()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

In [90]:
learning_rate = 1e-3
batch_size = 128
epochs = 10
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [91]:
#train and test loop

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        #runs forward prop through the model
        pred = model(X)
        #computes the cross entropy loss of the models prediction vs the label
        loss = loss_fn(pred, y)

        #running back prop
        loss.backward()
        optimizer.step()
        #clears grad and frees memory
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            #weird regex lol, copied from pytorch docs
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    #intentionally disabling the gradient to speed up computation
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [92]:
#running the loop

loss_fn = nn.CrossEntropyLoss()

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dl, model, loss_fn, optimizer)
    test_loop(test_dl, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.332832  [  128/60000]
loss: 2.276138  [12928/60000]


KeyboardInterrupt: 